### 一、通俗比喻
* 进程 = 一个独立的工厂<br>
每个工厂有自己的厂房、机器、工人（资源），工厂之间完全隔离，互不影响。
如果要合作（比如传递半成品），必须通过复杂的外部管道（进程间通信，IPC）。

* 线程 = 工厂里的一条生产线<br>
一个工厂可以有多条生产线，它们共享厂里的原料、仓库（进程的内存、文件等）。
生产线之间切换由工厂调度（操作系统调度线程），切换成本中等。
Python 中由于有一把“大锁”（GIL），同一时间只有一条生产线能真正干活（多线程无法并行计算），但遇到等待原料（I/O）时，可以切到另一条线。

* 协程 = 生产线上的一个工人<br>
工人可以灵活地在多个任务之间切换：比如在等待机器加热的几秒钟里，他先去做另一件手工活。
这个切换完全由工人自己控制（用户态切换），不需要工厂调度，成本极低。
一个工人（单线程）就能高效处理很多小任务，尤其适合大量等待（网络、磁盘等）。

### 二、区别与联系一览
|     | 进程 | 线程 | 协程 |
|-----|-----|-----|-----|
| 资源 | 独立内存、文件等 | 共享所属进程的资源 | 共享线程的资源 |
| 切换成本 | 很高（切换工厂） | 中等（切换生产线） | 极低（换手上活） |
| 调度者 | 操作系统 | 操作系统 | 程序员 / Python 事件循环 |
| 数据共享 | 困难（需要 IPC） | 容易（但需加锁） | 容易（同线程天然安全） |
| Python 特性 | 可绕过 GIL 实现并行 | GIL 限制 CPU 并行，适合 I/O | 单线程内极高并发 I/O |

* 联系：
一个进程可以包含多个线程，一个线程可以包含多个协程。<br>
协程运行在线程内，线程运行在进程内。<br>

### 三、Python 代码示例
以下用 模拟 I/O 等待 来展示三种方式的并发效果（等待 1 秒，重复 3 次）。

#### 1. 同步方式（无并发）—— 作为对比

In [1]:
import time

def task(name):
    print(f"{name} 开始")
    time.sleep(1)          # 模拟 I/O 等待
    print(f"{name} 结束")

start = time.time()
for i in range(3):
    task(f"任务{i}")
print(f"同步耗时: {time.time()-start:.2f}秒")
# 输出：耗时约 3 秒（依次等待）

任务0 开始
任务0 结束
任务1 开始
任务1 结束
任务2 开始
任务2 结束
同步耗时: 3.00秒


#### 2. 多进程

In [ ]:
import time
from multiprocessing import Pool

def task(name):
    print(f"{name} 开始")
    time.sleep(1)
    print(f"{name} 结束")

if __name__ == "__main__":
    start = time.time()
    with Pool(3) as p:
        p.map(task, [f"进程{i}" for i in range(3)])
    print(f"多进程耗时: {time.time()-start:.2f}秒")
# 输出：耗时约 1 秒（三个工厂同时干活）

### 3. 多线程

In [2]:
import time
import threading

def task(name):
    print(f"{name} 开始")
    time.sleep(1)
    print(f"{name} 结束")

threads = []
start = time.time()
for i in range(3):
    t = threading.Thread(target=task, args=(f"线程{i}",))
    t.start()
    threads.append(t)
for t in threads:
    t.join()
print(f"多线程耗时: {time.time()-start:.2f}秒")
# 输出：耗时约 1 秒（三条生产线同时等待）

线程0 开始
线程1 开始
线程2 开始
线程0 结束
线程1 结束
线程2 结束
多线程耗时: 1.00秒


### 4. 协程（asyncio）

In [4]:
import asyncio
import time

async def task(name):
    print(f"{name} 开始")
    await asyncio.sleep(1)   # 模拟异步 I/O 等待
    print(f"{name} 结束")

async def main():
    await asyncio.gather(
        task("协程1"),
        task("协程2"),
        task("协程3"),
    )

start = time.time()
# asyncio.run(main())
await main()
print(f"协程耗时: {time.time()-start:.2f}秒")
# 输出：耗时约 1 秒（一个工人来回切换任务）

协程1 开始
协程2 开始
协程3 开始
协程1 结束
协程2 结束
协程3 结束
协程耗时: 1.01秒


### 四、何时选用哪个？
* CPU 密集型（大量计算） → 使用多进程，绕过 GIL，真正利用多核。
* I/O 密集型（网络请求、读写文件）且需要高并发 → 协程（开销最小，写法简洁）。
* I/O 密集型但需要调用现有的同步库（如 requests） → 多线程（简单，无需改造代码）。
* 需要稳定隔离（一个挂了不影响另一个） → 多进程。